# Paired C/D: filtered literature retrieval
Select T4 and run all four code cells. Reuses all C and unchanged D caches, including failures. Only five D inputs change. This is development validation, not a final test. No guarantee of better accuracy or shorter responses.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Select Runtime > Disconnect and delete runtime, reconnect and rerun this cell. Complete Google authorisation. No diagnosis has started.') from exc
if not (ROOT / 'Files/phm_filtered_selection_pilot.py').is_file():
    raise FileNotFoundError(f'Drive is connected but the updated project is missing at {ROOT}. Finish sync or locate CP5403 in the Files panel. Do not force-remount.')
os.chdir(ROOT)
EXPECTED = {'phm_condition_d.py': 'e22457642e68013b1cfd4201bb6951630711d7d4890fdb28659d70dafa7a80c6', 'phm_selection_pilot.py': '3eaef1b55c56a3b6a2977d564b54cc0ebdd0deab1548680072c2f569c29eeafb', 'phm_filtered_selection_pilot.py': 'f154c9c522426007b934f7ea21f43b51f662cb83942163f7770a2426016baf0a', 'phm_evidence_selection.py': 'bb8cd412496f3d055d11b6d0f4ac47e970f24d43b4c19493d9ff57fca8b4fac9', 'phm_literature_index.py': '725709141868ae494f1c515f67df5aec4451c0d137d461b3991c7ba436fb7f37', 'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9', 'test_phm_selection.py': '054c50d83c00c2cc088f9cd93be4f9cb1f35b22346115bc62254a84a58ccd4d0', 'test_phm_filtered_retrieval.py': '628444b8017a5d52c7e3e1a3de21411c2e323070045ca7e43ea51bb2947986d9', 'literature_retrieval_policy_v1.json': '728102ca47c5074f43f7c2ea37a19bbcf0d9a91d896dde530ccf77b8d522ab33'}
for name, expected in EXPECTED.items():
    path = ROOT / 'Files' / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected, f'File not fully synced: {name}'
INDEX = ROOT / 'Datasets/PHM15-literature-D-approved-v1'
manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
for name, key in [('approved_chunks.jsonl','chunks_sha256'), ('embeddings.npy','embeddings_sha256'), ('model_assets/minilm.onnx','onnx_sha256')]:
    assert hashlib.sha256((INDEX / name).read_bytes()).hexdigest() == manifest[key], f'Index not fully synced: {name}'
print('Project and approved index verified:', manifest['chunk_count'], 'chunks')


Mounted at /content/drive
Project and approved index verified: 24 chunks


In [2]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-literature-colab.txt', 'onnxruntime==1.31.1', 'lm-format-enforcer==0.11.3'], check=True)
print('Dependencies installed.')


GPU: Tesla T4
Dependencies installed.


In [3]:
for test in ['test_phm_diagnosis.py', 'test_phm_selection.py', 'test_phm_filtered_retrieval.py']:
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', test, '-q'], check=True)
subprocess.run([sys.executable, 'Files/phm_filtered_selection_pilot.py', '--root', str(ROOT), '--max-cases', '10', '--prepare-only'], check=True)
print('Paired evidence and source checks passed. The final cell runs Qwen.')

sys.path.insert(0, str(ROOT / 'Files'))
import phm_filtered_selection_pilot as filtered
cfg = json.loads((ROOT/'Files/experiment_config.json').read_text(encoding='utf-8'))
cfg['generator']['max_input_tokens'] = 6144
_, records, cs, ds, _ = filtered.prepare(ROOT, cfg, 10)
cached = 0
for request, record in zip(cs + ds, records + records):
    path = ROOT/'Outputs/diagnosis'/request['condition']/'responses'/(request['request_id'] + '.json')
    check, response = filtered.inspect_c_paired(path, request, record)
    cached += response is not None and check['status'] != 'invalid_cache'
assert cached >= 15, f'Only {cached}/20 cached attempts verified. Finish syncing Outputs before continuing; do not regenerate the baseline.'
print('Verified cached attempts:', cached, 'New generations:', 20-cached)


Paired evidence and source checks passed. The final cell runs Qwen.
Verified cached attempts: 15 New generations: 5


In [4]:
subprocess.run([sys.executable, 'Files/phm_filtered_selection_pilot.py', '--root', str(ROOT), '--max-cases', '10'], check=True)
latest = json.loads((ROOT / 'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
print('Latest report:', latest)
print('Let Google Drive finish syncing, then ask for the paired C/D pilot review.')


Latest report: {'run_id': '20261122T001817_paired_CD_filtered_retrieval_pilot_699db3b8', 'run_directory': 'Outputs/runs/20261122T001817_paired_CD_filtered_retrieval_pilot_699db3b8', 'status': 'completed', 'updated_utc': '2026-11-22T00:24:47.969684+00:00'}
Let Google Drive finish syncing, then ask for the paired C/D pilot review.
